# Parse dos eventos brutos (JSON -> tipos estruturados)

O `extract_possession_events.ipynb` serializa `details`/`homePlayers`/`awayPlayers`/`balls` como string JSON antes de gravar em parquet (`normalize_columns`). Esse notebook lê esses parquets brutos, refaz o parse pra tipos estruturados nativos (struct/array/map) uma única vez, e grava uma versão "canônica" em `data/events_parsed/` — assim nenhum notebook de engenharia precisa repetir esse tratamento.

Também padroniza os nomes de coluna (`id` -> `eventId`, `player.id` -> `eventPlayerId`, etc.) e deriva `eventSubTypeDescription`/`eventOutcomeDescription` a partir do `details_parsed`, já que praticamente todo notebook downstream também precisa dessas duas colunas.

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [2]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .master("local[*]")
    .appName("parse_possession_events")
    .getOrCreate()
    )

In [3]:
data_dir = Path().resolve().parent.parent / "data"
raw_events_dir = data_dir / "events"
parsed_events_dir = data_dir / "events_parsed"

In [4]:
# Schemas usados pra parsear as colunas que a extração gravou como string JSON
players_schema = ArrayType(
    StructType([
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("player", StructType([
            StructField("name", StringType(), True)]),
            True),
    ])
)

balls_schema = ArrayType(
    StructType([
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("z", FloatType(), True),
    ])
)

details_schema = MapType(StringType(), StringType())

In [5]:
# ============================================================
# Chave de tipo/outcome por eventType dentro do details_parsed
# ============================================================
# Cada eventType guarda o "tipo" específico e o "outcome" dele numa chave
# diferente do map. Quando o eventType não tem uma dessas chaves (ex:
# Clearance não tem tipo, kickoff não tem outcome), fica NULL.

event_type_desc_key = (
    F.when(F.col('eventType').isin('OTB', 'FIRSTKICKOFF', 'SECONDKICKOFF'), F.lit('setpieceTypeDescription'))
    .when(F.col('eventType') == 'BC', F.lit('carryTypeDescription'))
    .when(F.col('eventType').isin('CH', 'FO'), F.lit('challengeTypeDescription'))
    .when(F.col('eventType') == 'CR', F.lit('crossTypeDescription'))
    .when(F.col('eventType') == 'PA', F.lit('passTypeDescription'))
    .when(F.col('eventType') == 'SH', F.lit('shotTypeDescription'))
    .when(F.col('eventType') == 'TC', F.lit('touchTypeDescription'))
    # CL (Clearance) e RE (Rebound) não têm chave de tipo no details
)

event_outcome_desc_key = (
    F.when(F.col('eventType') == 'BC', F.lit('ballCarryOutcomeDescription'))
    .when(F.col('eventType').isin('CH', 'FO'), F.lit('challengeOutcomeTypeDescription'))
    .when(F.col('eventType') == 'CL', F.lit('clearanceOutcomeTypeDescription'))
    .when(F.col('eventType') == 'CR', F.lit('crossOutcomeTypeDescription'))
    .when(F.col('eventType') == 'PA', F.lit('passOutcomeTypeDescription'))
    .when(F.col('eventType') == 'RE', F.lit('reboundOutcomeTypeDescription'))
    .when(F.col('eventType') == 'SH', F.lit('shotOutcomeTypeDescription'))
    .when(F.col('eventType') == 'TC', F.lit('touchOutcomeTypeDescription'))
    # OTB e os kickoffs não têm chave de outcome no details
)

In [6]:
def parse_events_folder(raw_folder: Path, parsed_folder: Path):
    """
    Lê todos os parquets brutos de uma pasta competitionId/season (com
    homePlayers/awayPlayers/balls/details serializados como string JSON)
    e grava a versão equivalente já parseada em parsed_folder.
    """
    parquet_files = [str(p) for p in raw_folder.glob("*.parquet")]
    if not parquet_files:
        return None

    df = spark.read.parquet(*parquet_files)

    df = df.withColumnsRenamed({
        "id": "eventId",
        "player.id": "eventPlayerId",
        "player.name": "eventPlayerName",
        "team.id": "eventTeamId",
        "team.name": "eventTeamName",
    })

    df = df.withColumns({
        "homePlayers_parsed": F.from_json("homePlayers", players_schema),
        "awayPlayers_parsed": F.from_json("awayPlayers", players_schema),
        "balls_parsed": F.from_json("balls", balls_schema),
        "details_parsed": F.from_json("details", details_schema),
    }).drop("homePlayers", "awayPlayers", "balls", "details")

    df = (
        df
        .withColumns({
            "eventSubTypeDescription": F.element_at(F.col("details_parsed"), event_type_desc_key),
            "eventOutcomeDescription": F.element_at(F.col("details_parsed"), event_outcome_desc_key),
        })
        #.drop("details_parsed")
    )

    parsed_folder.mkdir(parents=True, exist_ok=True)
    df.write.mode("overwrite").parquet(str(parsed_folder))

    return df

In [7]:
# Percorre todas as pastas competitionId/season já extraídas em data/events
# e grava a versão parseada equivalente em data/events_parsed
season_folders = sorted(p for p in raw_events_dir.glob("*/*") if p.is_dir())

for season_folder in season_folders:
    competition_id = season_folder.parent.name
    season = season_folder.name

    parsed_folder = parsed_events_dir / competition_id / season

    print(f"Parseando competitionId={competition_id}, season={season}...")
    result = parse_events_folder(season_folder, parsed_folder)

    if result is None:
        print(f"  -> nenhum parquet encontrado em {season_folder}, pulando.")
    else:
        print(f"  -> gravado em {parsed_folder}")

Parseando competitionId=1, season=2020-2021...
  -> gravado em C:\Users\MatheusSantos\Documents\Projetos\Mestrado\defensive-performance-prediction\data\events_parsed\1\2020-2021
Parseando competitionId=1, season=2021-2022...
  -> gravado em C:\Users\MatheusSantos\Documents\Projetos\Mestrado\defensive-performance-prediction\data\events_parsed\1\2021-2022
Parseando competitionId=1, season=2022-2023...
  -> gravado em C:\Users\MatheusSantos\Documents\Projetos\Mestrado\defensive-performance-prediction\data\events_parsed\1\2022-2023
Parseando competitionId=1, season=2023-2024...
  -> gravado em C:\Users\MatheusSantos\Documents\Projetos\Mestrado\defensive-performance-prediction\data\events_parsed\1\2023-2024
Parseando competitionId=1, season=2024-2025...
  -> gravado em C:\Users\MatheusSantos\Documents\Projetos\Mestrado\defensive-performance-prediction\data\events_parsed\1\2024-2025
Parseando competitionId=42, season=2023...
  -> gravado em C:\Users\MatheusSantos\Documents\Projetos\Mestrado\